In [ ]:
%pip install torch_geometric
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
import torch_geometric.transforms as T

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, f1_score

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "credit_card_transactions-ibm_v2.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ealtman2019/credit-card-transactions",
  file_path,
)

Using Colab cache for faster access to the 'credit-card-transactions' dataset.


In [ ]:
non_fraud = df[df['Is Fraud?'] == 'No']
fraud = df[df['Is Fraud?'] == 'Yes']

non_fraud_reduced = non_fraud.sample(n=len(non_fraud)-23000000, random_state=42)

df = pd.concat([fraud, non_fraud_reduced]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['Is Fraud?'].value_counts())

(1386900, 15)
Is Fraud?
No     1357143
Yes      29757
Name: count, dtype: int64


# Data cleaning

In [ ]:
# 1. Strip $ from Amount and convert to float
df['Amount'] = df['Amount'].str.replace('$', '', regex=False).astype(float)

# 2. Encode Is Fraud? to 0/1
df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})

# 3. Extract hour from Time
df['Hour'] = df['Time'].str.split(':').str[0].astype(int)
df.drop(columns=['Time'], inplace=True)

# map use chips
df['Use Chip'] = df['Use Chip'].map({'Online Transaction': 0, 'Swipe Transaction': 1, 'Chip Transaction': 2})

In [ ]:
df.tail(10)

,User,Card,Year,Month,Day,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,Hour,card_id,cust_node_idx,merch_node_idx,merchant_city,errors
1386890,1154,1,2007,3,13,106.37,1,-3971935829054965805,Revere,MA,2151.0,5300,No error,0,8,1154_1,3956,2036,7599,17
1386891,370,0,2007,5,2,32.36,0,-2088492411650162548,ONLINE,NaN,NaN,4784,No error,0,16,370_0,595,3,6547,17
1386892,1772,0,2014,1,7,8.77,1,7847619186796796084,Williamsburg,PA,16693.0,5812,No error,0,7,1772_0,214,868,9935,17
1386893,1151,2,2007,5,1,-92.00,1,1799189980464955940,Hamilton,OH,45013.0,5499,No error,0,7,1151_2,2542,41,3743,17
1386894,1123,2,2017,12,18,18.38,1,-4500542936415012428,San Antonio,TX,78211.0,5814,No error,0,13,1123_2,3435,154,8024,17
1386895,896,2,2013,5,9,9.25,1,1874515938428798953,Louisville,OH,44641.0,5411,No error,0,6,896_2,248,1231,5224,17
1386896,1488,1,2018,6,2,143.97,2,-5467922351692495955,Appleton,WI,54915.0,5912,No error,0,11,1488_1,1459,78,239,17
1386897,697,0,2012,5,13,94.16,1,1913477460590765860,Vienna,VA,22182.0,5300,No error,0,16,697_0,1789,42,9433,17
1386898,597,1,2011,11,25,-91.00,1,1799189980464955940,Atlanta,GA,30318.0,5499,No error,0,9,597_1,2241,41,341,17
1386899,777,5,2005,4,9,80.64,1,-8254405722253003769,Murdock,MN,56271.0,5499,No error,0,10,777_5,979,244,6119,17


In [ ]:
df["card_id"] = df["User"].astype(str) + "_" + df["Card"].astype(str)
df.drop(columns=['Merchant State', 'Card', 'User', 'Zip'])
df["Errors?"] = df["Errors?"].fillna("No error")

In [ ]:
customer_ids = df["card_id"].unique()
merchant_ids = df["Merchant Name"].unique()

customer_id_map = {cid: i for i, cid in enumerate(customer_ids)}
merchant_id_map = {mid: i for i, mid in enumerate(merchant_ids)}

df["cust_node_idx"] = df["card_id"].map(customer_id_map)
df["merch_node_idx"] = df["Merchant Name"].map(merchant_id_map)


# Building the Graph

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns to numeric for PyTorch
le = LabelEncoder()
df['merchant_city'] = le.fit_transform(df['Merchant City'].astype(str))
df['errors'] = le.fit_transform(df['Errors?'].astype(str))

# Updated feature list with numeric columns only
feature_cols = ["Amount", "Hour", "Use Chip", "merchant_city", "errors", "MCC"]
transaction_x = torch.tensor(df[feature_cols].values.astype(np.float32), dtype=torch.float)

# customer node features: simple aggregates over that customer's transactions
cust_agg = df.groupby("card_id")["Amount"].agg(["mean", "count"]).reindex(customer_ids).fillna(0)
customer_x = torch.tensor(cust_agg.values.astype(np.float32), dtype=torch.float)

# merchant node features: same idea
merch_agg = df.groupby("Merchant Name")["Amount"].agg(["mean", "count"]).reindex(merchant_ids).fillna(0)
merchant_x = torch.tensor(merch_agg.values.astype(np.float32), dtype=torch.float)

In [ ]:
num_txns = len(df)
txn_idx = np.arange(num_txns)

cust_to_txn = torch.tensor(
    np.vstack([df["cust_node_idx"].values, txn_idx]), dtype=torch.long
)
merch_to_txn = torch.tensor(
    np.vstack([df["merch_node_idx"].values, txn_idx]), dtype=torch.long
)

In [ ]:
data = HeteroData()

data["customer"].x = customer_x
data["merchant"].x = merchant_x
data["transaction"].x = transaction_x
data["transaction"].y = torch.tensor(df["Is Fraud?"].values, dtype=torch.float)

data["customer", "made", "transaction"].edge_index = cust_to_txn
data["merchant", "involved_in", "transaction"].edge_index = merch_to_txn

# add the reverse-direction edges automatically, so info can flow transaction -> customer too
data = T.ToUndirected()(data)

print(data)

HeteroData(
  customer={ x=[5971, 2] },
  merchant={ x=[38892, 2] },
  transaction={
    x=[1386900, 6],
    y=[1386900],
  },
  (customer, made, transaction)={ edge_index=[2, 1386900] },
  (merchant, involved_in, transaction)={ edge_index=[2, 1386900] },
  (transaction, rev_made, customer)={ edge_index=[2, 1386900] },
  (transaction, rev_involved_in, merchant)={ edge_index=[2, 1386900] }
)


# Train/Test split

In [ ]:
# Temporal split based on 'Year'
split_year = df['Year'].quantile(0.9)
print(f"Splitting data at year: {int(split_year)}")

train_mask = torch.from_numpy((df['Year'] < split_year).values)
test_mask = torch.from_numpy((df['Year'] >= split_year).values)

data["transaction"].train_mask = train_mask
data["transaction"].test_mask = test_mask

print(f"Training samples: {train_mask.sum().item()}")
print(f"Testing samples: {test_mask.sum().item()}")
print(train_mask)
print(test_mask)
print(data)
print("*******")
node_types, edge_types = data.metadata()
print("Node types:", node_types)
print("Edge types:", edge_types)

Splitting data at year: 2018
Training samples: 1171626
Testing samples: 215274
tensor([ True,  True, False,  ...,  True,  True,  True])
tensor([False, False,  True,  ..., False, False, False])
HeteroData(
  customer={ x=[5971, 2] },
  merchant={ x=[38892, 2] },
  transaction={
    x=[1386900, 6],
    y=[1386900],
    train_mask=[1386900],
    test_mask=[1386900],
  },
  (customer, made, transaction)={ edge_index=[2, 1386900] },
  (merchant, involved_in, transaction)={ edge_index=[2, 1386900] },
  (transaction, rev_made, customer)={ edge_index=[2, 1386900] },
  (transaction, rev_involved_in, merchant)={ edge_index=[2, 1386900] }
)
*******
Node types: ['customer', 'merchant', 'transaction']
Edge types: [('customer', 'made', 'transaction'), ('merchant', 'involved_in', 'transaction'), ('transaction', 'rev_made', 'customer'), ('transaction', 'rev_involved_in', 'merchant')]


# Model


In [ ]:
class HeteroSAGE(nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata):
        super().__init__()
        edge_types = metadata[1]
        self.conv1 = HeteroConv(
            {et: SAGEConv((-1, -1), hidden_channels) for et in edge_types}, aggr="sum"
        )
        self.conv2 = HeteroConv(
            {et: SAGEConv((-1, -1), out_channels) for et in edge_types}, aggr="sum"
        )
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict


In [ ]:
class FraudHead(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.lin = nn.Linear(in_channels, 1)

    def forward(self, transaction_emb):
        return self.lin(transaction_emb).squeeze(-1)

## Training

In [ ]:
  encoder = HeteroSAGE(hidden_channels=64, out_channels=32, metadata=data.metadata())
  head = FraudHead(32)

  # Calculate pos_weight: non-fraud / fraud = 1357143 / 29757 ≈ 45.6
  pos_weight = torch.tensor([1357143 / 29757])

  optimizer = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=0.01)
  criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

  y = data["transaction"].y
  train_mask = data["transaction"].train_mask

  print(f"Starting training with pos_weight: {pos_weight.item():.2f}")

  for epoch in range(70):
      encoder.train(); head.train()
      optimizer.zero_grad()

      out_dict = encoder(data.x_dict, data.edge_index_dict)
      logits = head(out_dict["transaction"])

      # Training loss calculation
      loss = criterion(logits[train_mask], y[train_mask])
      loss.backward()
      optimizer.step()

      if epoch % 10 == 0:
          # Simple debug metrics for training set
          with torch.no_grad():
              preds = (logits[train_mask] > 0).float()
              correct = (preds == y[train_mask]).sum().item()
              acc = correct / train_mask.sum().item()
              fraud_preds = preds.sum().item()

          print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Train Acc: {acc:.4f} | Fraud Preds: {int(fraud_preds)}")

# Retrieve embeddings

In [ ]:
encoder.eval()
with torch.no_grad():
    out_dict = encoder(data.x_dict, data.edge_index_dict)

transaction_embeddings = out_dict["transaction"].numpy()  # shape: [num_txns, 32]

emb_df = pd.DataFrame(
    transaction_embeddings,
    columns=[f"gnn_emb_{i}" for i in range(transaction_embeddings.shape[1])]
)
# row order of emb_df matches row order of df, since df.index built transaction_x
df_with_embeddings = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

emb_df.head()

# Merge embeddigns with original data

In [ ]:
df = df.reset_index(drop=True)
emb_df = emb_df.reset_index(drop=True)

df = pd.concat([df, emb_df], axis=1) # MUST CORRESPOND PERFECTLY TO AVOID GARBAGE
df.head()

In [ ]:
X = df.drop(columns=['Is Fraud?'])
y = df['Is Fraud?']

# these masks came from the GNN pipeline -- same boolean arrays, same row order as df
train_mask_np = data["transaction"].train_mask.numpy()
test_mask_np = data["transaction"].test_mask.numpy()

X_train, X_test = X[train_mask_np], X[test_mask_np]
y_train, y_test = y[train_mask_np], y[test_mask_np]

print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
# 6. scale_pos_weight from training data
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

# 7. Train XGBoost
# Drop non-numeric columns that XGBoost cannot handle
non_numeric_cols = X_train.select_dtypes(include=['object']).columns
X_train_numeric = X_train.drop(columns=non_numeric_cols)
X_test_numeric = X_test.drop(columns=non_numeric_cols)

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    verbosity=2,  # Set verbosity to 2 for more info
    random_state=42
)

# Adding eval_set to see progress during training
model.fit(
    X_train_numeric,
    y_train,
    eval_set=[(X_train_numeric, y_train), (X_test_numeric, y_test)],
    verbose=True
)

# Evaluate

In [ ]:
y_pred = model.predict(X_test_numeric)
y_proba = model.predict_proba(X_test_numeric)[:, 1]

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

In [ ]:
print(df.head())

In [ ]:
# ── Classify a mock transaction ──────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

le_city = LabelEncoder().fit(df['Merchant City'].astype(str))
le_err  = LabelEncoder().fit(df['Errors?'].astype(str))

def classify_transaction(txn):
    # 1. node lookup (new customer/merchant -> zero-init aggregates)
    cust_idx = customer_id_map.get(txn["card_id"], len(customer_id_map))
    merch_idx = merchant_id_map.get(txn["merchant_name"], len(merchant_id_map))
    new_cust, new_merch = cust_idx == len(customer_id_map), merch_idx == len(merchant_id_map)

    city_enc = le_city.transform([txn["merchant_city"]])[0] if txn["merchant_city"] in le_city.classes_ else -1
    err_enc  = le_err.transform([txn["errors"]])[0] if txn["errors"] in le_err.classes_ else -1

    feat = torch.tensor([[txn["amount"], txn["hour"], txn["use_chip"], city_enc, err_enc, txn["mcc"]]], dtype=torch.float)

    # 2. splice into a copy of the graph
    g = data.clone()
    g["transaction"].x = torch.cat([g["transaction"].x, feat])
    new_idx = g["transaction"].x.shape[0] - 1
    if new_cust: g["customer"].x = torch.cat([g["customer"].x, torch.zeros(1, g["customer"].x.shape[1])])
    if new_merch: g["merchant"].x = torch.cat([g["merchant"].x, torch.zeros(1, g["merchant"].x.shape[1])])

    def link(et, src): g[et].edge_index = torch.cat([g[et].edge_index, torch.tensor([[src],[new_idx]])], dim=1)
    link(("customer","made","transaction"), cust_idx)
    link(("merchant","involved_in","transaction"), merch_idx)
    g = T.ToUndirected()(g)

    # 3. rerun GNN -> get embedding for just this transaction
    encoder.eval()
    with torch.no_grad():
        emb = encoder(g.x_dict, g.edge_index_dict)["transaction"][new_idx].numpy()

    # 4. build XGBoost row and predict
    row = {"Amount": txn["amount"], "Hour": txn["hour"], "Use Chip": txn["use_chip"],
           "merchant_city": city_enc, "errors": err_enc, "MCC": txn["mcc"]}
    row.update({f"gnn_emb_{i}": v for i, v in enumerate(emb)})
    X_row = pd.DataFrame([row]).reindex(columns=X_train_numeric.columns, fill_value=0)

    proba = model.predict_proba(X_row)[0, 1]
    return {"fraud_probability": float(proba), "prediction": "FRAUD" if proba >= 0.5 else "LEGITIMATE"}


# Configure your mock transaction here
mock_txn = {
    "card_id": df["card_id"].iloc[0],
    "merchant_name": "NEW_MERCHANT_XYZ",
    "amount": 4500.0,
    "hour": 3,
    "use_chip": 0,       # 0=Online, 1=Swipe, 2=Chip
    "merchant_city": "ONLINE",
    "errors": "No error",
    "mcc": 5732,
}
print(classify_transaction(mock_txn))